# D1 Utilities Workflow — GeoPandas

Utility network analysis using GeoPandas spatial operations.

In [14]:
from __future__ import annotations
import json, os
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_LIVE_API = os.getenv("GEOPROMPT_ALLOW_LIVE_API", "0") == "1"

def fetch_json(url, fallback):
    if not ALLOW_LIVE_API:
        return fallback
    try:
        req = Request(url, headers={"User-Agent": "geoprompt-notebook/2.0"})
        with urlopen(req, timeout=10) as r:
            return json.loads(r.read().decode("utf-8"))
    except (URLError, TimeoutError, ValueError):
        return fallback

import geopandas as gpd
from shapely.geometry import Point, Polygon, box
print(f"geopandas {gpd.__version__} ready")


geopandas 1.1.3 ready


## Section A: Pull Data Sources

In [15]:
eia   = fetch_json("https://api.eia.gov/v2/electricity/facility-fuel/data/", {"response": {"data": []}})
openei = fetch_json("https://openei.org/services/rest/search?api_key=DEMO&q=utility", {"result": []})
print("EIA records:", len(eia.get("response", {}).get("data", [])))
print("OpenEI results:", len(openei.get("result", [])))


EIA records: 0
OpenEI results: 0


## Section B: Spatial Analysis (GeoPandas)

In [18]:
# Utility substation data
nodes_data = {
    "node":   ["PLANT", "SUB1", "SUB2", "SUB3", "SUB4", "SUB5", "SUB6"],
    "cost":   [0.0,     3.0,    8.0,    11.0,   7.0,    4.0,    13.0],
    "cap_mw": [500,     120,    90,      80,    110,    100,     70],
    "type":   ["plant", "sub",  "sub",   "sub",  "sub",  "sub",  "sub"],
}
lons = [-87.70, -87.63, -87.72, -87.65, -87.58, -87.76, -87.60]
lats = [ 41.94,  41.88,  41.83,  41.79,  41.85,  41.91,  41.75]

# 1. Build GeoDataFrame
gdf = gpd.GeoDataFrame(nodes_data, geometry=gpd.points_from_xy(lons, lats), crs="EPSG:4326")
print("GeoDataFrame:")
print(gdf[["node", "cost", "cap_mw"]].to_string(index=False))

# 2. Buffer: service zones around substations
gdf_projected = gdf.to_crs("EPSG:3857")
gdf_projected["buffer_geom"] = gdf_projected.buffer(5000)  # 5 km buffer
print(f"\nBuffer zones (5 km): {len(gdf_projected)} polygons")

# 3. Bounding box filter: cx selector for northeast quadrant
ne_gdf = gdf.cx[-87.72:-87.58, 41.83:41.95]
print(f"\nNodes in NE quadrant: {list(ne_gdf['node'])}")

# 4. Nearest join: find closest substation to each demand point (in km).
demand_data = {"demand_id": ["D1","D2","D3"], "load_mw": [45, 70, 30]}
demand_lons = [-87.66, -87.75, -87.61]
demand_lats = [ 41.87,  41.86,  41.81]
demand_gdf = gpd.GeoDataFrame(demand_data, geometry=gpd.points_from_xy(demand_lons, demand_lats), crs="EPSG:4326")
demand_m = demand_gdf.to_crs("EPSG:3857")
substations_m = gdf[gdf["type"] == "sub"].to_crs("EPSG:3857")
nearest_m = gpd.sjoin_nearest(demand_m, substations_m, how="left", distance_col="dist_m")
nearest = nearest_m.to_crs("EPSG:4326")
nearest["dist_km"] = nearest_m["dist_m"] / 1000.0
print("\nNearest substation assignments (km):")
print(nearest[["demand_id", "node", "dist_km"]].to_string(index=False))

# 5. Spatial join: demand points within substation buffers
gdf_buf = gpd.GeoDataFrame(gdf[["node"]], geometry=gdf.to_crs("EPSG:3857").buffer(8000).to_crs("EPSG:4326"))
sj = gpd.sjoin(demand_gdf, gdf_buf[["node","geometry"]], how="left", predicate="within")
print("Demand within buffer zones:")
print(sj[["demand_id", "node"]].to_string(index=False))

# 6. Dissolve by node type: aggregate capacity
dissolved = gdf.dissolve(by="type", aggfunc={"cap_mw": "sum", "cost": "mean"})
print("\nDissolved by type:")
print(dissolved[["cap_mw", "cost"]].to_string())

# 7. High-cost nodes filter
high_cost = gdf[gdf["cost"] > 7.0]
print(f"\nHigh-cost nodes (cost > 7): {list(high_cost['node'])}")

# Save to file
gdf.to_file(str(OUTPUT_DIR / "d1-gpd-nodes.geojson"), driver="GeoJSON")
print("\nWrote d1-gpd-nodes.geojson")

# Inline visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gdf.plot(ax=axes[0], column="cost", cmap="RdYlGn_r", markersize=120, legend=True,
         legend_kwds={"label": "Routing Cost"})
for _, row in gdf.iterrows():
    axes[0].annotate(row["node"], (row.geometry.x, row.geometry.y),
                     textcoords="offset points", xytext=(4, 4), fontsize=8)
demand_gdf.plot(ax=axes[0], color="blue", markersize=80, marker="D", label="Demand")
axes[0].set_title("Utility Nodes (diamonds=demand points)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

gdf_sorted = gdf.sort_values("cost")
axes[1].barh(gdf_sorted["node"], gdf_sorted["cost"], color="#2563eb")
axes[1].set_xlabel("Routing Cost"); axes[1].set_title("Node Costs")
axes[1].grid(True, axis="x", alpha=0.3)
plt.suptitle("D1 Utilities: GeoPandas Spatial Analysis", fontweight="bold")
plt.tight_layout(); plt.show()


GeoDataFrame:
 node  cost  cap_mw
PLANT   0.0     500
 SUB1   3.0     120
 SUB2   8.0      90
 SUB3  11.0      80
 SUB4   7.0     110
 SUB5   4.0     100
 SUB6  13.0      70

Buffer zones (5 km): 7 polygons

Nodes in NE quadrant: ['PLANT', 'SUB1', 'SUB2', 'SUB4']

Nearest substation assignments (km):
demand_id node  dist_km
       D1 SUB1 3.658949
       D2 SUB2 5.590141
       D3 SUB3 5.361589
Demand within buffer zones:
demand_id node
       D1 SUB1
       D2 SUB2
       D2 SUB5
       D3 SUB3
       D3 SUB4

Dissolved by type:
       cap_mw      cost
type                   
plant     500  0.000000
sub       570  7.666667

High-cost nodes (cost > 7): ['SUB2', 'SUB3', 'SUB6']

Wrote d1-gpd-nodes.geojson


C:\Users\Matt\AppData\Local\Temp\ipykernel_21360\1503980691.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Section C: Scenario Comparison

In [17]:
scenarios = {
    "baseline":  {"reliability": 0.88, "outage_hrs_yr": 12.0, "cost_musd": 0.0},
    "hardened":  {"reliability": 0.94, "outage_hrs_yr":  5.0, "cost_musd": 28.0},
    "redundant": {"reliability": 0.97, "outage_hrs_yr":  2.5, "cost_musd": 55.0},
}
scenario_records = []
for name, vals in scenarios.items():
    score = round(vals["reliability"] * 0.5 + (1.0 / max(vals["outage_hrs_yr"], 0.1)) * 3 * 0.3
                  + (1.0 / max(vals["cost_musd"] + 1, 1)) * 20 * 0.2, 4)
    scenario_records.append({"scenario": name, **vals, "score": score})
scenario_records.sort(key=lambda r: -float(r["score"]))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
names = [r["scenario"] for r in scenario_records]
colors = ["#27ae60", "#e67e22", "#c0392b"]
axes[0].barh(names, [r["reliability"] for r in scenario_records], color=colors)
axes[0].set_xlabel("Reliability"); axes[0].set_title("Network Reliability"); axes[0].grid(True, axis="x", alpha=0.3)
axes[1].barh(names, [r["outage_hrs_yr"] for r in scenario_records], color=colors)
axes[1].set_xlabel("Outage Hours/Year"); axes[1].set_title("Annual Outages"); axes[1].grid(True, axis="x", alpha=0.3)
axes[2].barh(names, [r["score"] for r in scenario_records], color=colors)
axes[2].set_xlabel("Composite Score"); axes[2].set_title("Scenario Score"); axes[2].grid(True, axis="x", alpha=0.3)
plt.suptitle("D1 Utilities (GeoPandas): Scenario Comparison", fontweight="bold")
plt.tight_layout(); plt.show()

(OUTPUT_DIR / "d1-gpd-complex.json").write_text(
    json.dumps({"scenario_ranking": scenario_records}, indent=2, default=str), encoding="utf-8"
)
print("Wrote d1-gpd-complex.json")
for r in scenario_records:
    print(f"  {r['scenario']}: score={r['score']}")


Wrote d1-gpd-complex.json
  baseline: score=4.515
  redundant: score=0.9164
  hardened: score=0.7879


C:\Users\Matt\AppData\Local\Temp\ipykernel_21360\3669264877.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
